# DW-1 Sprint 4: внешний LLM-перефраз + Fast-DetectGPT

Только внешний CUDA-контур и **отдельно разрешённый пакет для `sprint4-attacks-v2`**.
Разрешение на BCubed сюда не переносится. Сырые письма, перефразы и страницу проверки не публикуйте.

1. Выберите **Среда выполнения → Сменить среду → T4 GPU**.
2. Локально соберите `external_bundle.zip` по инструкции задачи 2 и загрузите его ниже.
3. Выполните ячейки; это долгий прогон без локального torch. Никакого подключения Google Drive.
4. Скачайте приватный `external_results.zip`. Откройте `semantic_review.html` и проверьте смысл/язык всех перефразов человеком.
5. Верните результаты и заполненный `semantic_review.jsonl` в локальный `make eval ATTACKS_V2=1 ...`.

Этот ноутбук **не содержит 600 писем** и не выдаёт тестовые примеры за реальный результат.
Параметр strength меняет инструкцию о глубине перефраза; реальная степень изменений не гарантирована.
Документация: [Qwen](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct), [Transformers generation](https://huggingface.co/docs/transformers/main_classes/text_generation).


In [ ]:
# Не фиксируем numpy==1.26: для Python 3.13 это приводит к сборке из исходников.
# Только готовые wheels. CUDA torch берётся из внешней среды Colab.
%pip install --quiet --upgrade --only-binary=:all: "transformers>=4.46,<5" "pandas>=2.2" "pyarrow>=18" "safetensors>=0.4" "huggingface_hub>=0.26,<1"


In [ ]:
import importlib.util
import stat
import zipfile
from pathlib import Path

from google.colab import files

uploaded = files.upload()  # Загрузите только external_bundle.zip, без моделей joblib.
if len(uploaded) != 1:
    raise ValueError("Нужен ровно один подготовленный external_bundle.zip")
archive = next(iter(uploaded))
package = Path('/content/dw1_attacks_package')
if package.exists():
    raise ValueError("Папка уже существует. Для повторного запуска используйте готовую папку или новую сессию.")
package.mkdir()
with zipfile.ZipFile(archive) as z:
    for item in z.infolist():
        if Path(item.filename).name != item.filename or '\\' in item.filename or item.is_dir():
            raise ValueError("Пакет должен иметь плоскую структуру без внешних путей")
        if stat.S_ISLNK(item.external_attr >> 16):
            raise ValueError("Ссылки в ZIP запрещены")
        if item.file_size > 200_000_000:
            raise ValueError("Неожиданный размер файла")
    z.extractall(package)
# Исполняется только аудированный снимок run_attacks_v2.py из подготовленного вами пакета.
spec = importlib.util.spec_from_file_location('run_attacks_v2', package / 'run_attacks_v2.py')
runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)
manifest, variants, requests = runner.load_package(package)
print('Package ID:', manifest['package_id'])
print('Messages:', manifest['n_messages'], '| variants:', len(variants), '| LLM requests:', len(requests))
print('External approval:', manifest.get('external_allowed'), '| purpose:', manifest.get('purpose'))


In [ ]:
import torch
from huggingface_hub import HfApi

if not torch.cuda.is_available():
    raise RuntimeError('Выберите T4 GPU; локальный запуск запрещён')
PARAPHRASE_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
DETECTOR_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
# Разрешаем revision один раз, затем запускаем обе модели по неизменяемым SHA.
# Для повторения ранее выполненного прогона подставьте SHA из run_config.json.
PARAPHRASE_REVISION = HfApi().model_info(PARAPHRASE_MODEL).sha
DETECTOR_REVISION = HfApi().model_info(DETECTOR_MODEL).sha
print('Generator:', PARAPHRASE_MODEL, PARAPHRASE_REVISION)
print('Detector:', DETECTOR_MODEL, DETECTOR_REVISION)
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
out = Path('/content/dw1_attacks_results')
runner.main([
    '--package', str(package), '--out', str(out),
    '--paraphrase-model', PARAPHRASE_MODEL, '--paraphrase-revision', PARAPHRASE_REVISION,
    '--detector-model', DETECTOR_MODEL, '--detector-revision', DETECTOR_REVISION,
    '--max-input-tokens', '4096', '--max-new-tokens', '2048', '--max-score-tokens', '4096',
    '--seed', '42',
])
# Результаты сохраняются постепенно. При разрыве GPU-сессии снова запустите эту
# ячейку с той же папкой/параметрами. Для новой сессии перенесите приватные checkpoints.
# Короткие или слишком длинные тексты не отбрасываются молча: прогон станет incomplete.


In [ ]:
# Выполняйте также после ошибки, чтобы скачать checkpoints и причины incomplete.
# Не публиковать: внутри исходные/перефразированные тексты и приватная HTML-проверка.
import shutil

result_zip = shutil.make_archive('/content/external_results', 'zip', root_dir=out)
files.download(result_zip)
print('Далее: semantic_review.html → semantic_review.jsonl → локальный make eval.')
